# Random Disclosure Parity Game: Q-Learning Agents

## Game Definition

> *The Random Disclosure Parity Game is a two-player zero-sum game with four moves:*
>
> 1. **Player I** selects an integer from $\{1, 2\}$.
> 2. **Referee** tosses a fair coin. Heads $\to$ Player II is informed of Player I's choice; Tails $\to$ Player II receives no information.
> 3. **Player II** selects an integer from $\{3, 4\}$.
> 4. **Referee** draws an integer from $\{1, 2, 3\}$ with probabilities $\{0.4, 0.2, 0.4\}$.
>
> The sum of moves 1, 3, and 4 is computed. If even, Player II pays Player I \$1; if odd, Player I pays Player II \$1.

This game features **imperfect information** (Player II may or may not know Player I's choice) and **chance moves** (coin toss, referee draw). We train Q-learning agents through self-play and verify they discover the known Nash equilibrium:

| Quantity | Value |
|:---------|:------|
| Game value | $v = -0.30$ (Player II has a \$0.30 edge) |
| Player I | Indifferent: mix $\{1, 2\}$ uniformly |
| Player II at $H_1$ | Play 3 (match parity) |
| Player II at $H_2$ | Play 4 (match parity) |
| Player II at $T$ | Indifferent: mix $\{3, 4\}$ uniformly |

### Why Softmax Q-Learning?

Standard $\varepsilon$-greedy Q-learning converges to **deterministic** policies. However, this game's Nash equilibrium requires **mixed strategies** (Player I and Player II at $T$ must randomise). We therefore use **Boltzmann (softmax) exploration** at a fixed temperature $\tau$, which naturally represents stochastic policies. The key observation is that while the instantaneous Q-values may oscillate, the **time-averaged empirical action frequencies** converge to the equilibrium mixing probabilities.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import random
from tqdm import tqdm

plt.style.use('dark_background')
plt.rcParams['font.family'] = 'monospace'
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.facecolor'] = '#1a1a2e'
plt.rcParams['figure.facecolor'] = '#0f0f23'
plt.rcParams['axes.edgecolor'] = '#4a4a6a'
plt.rcParams['axes.labelcolor'] = '#e0e0e0'
plt.rcParams['xtick.color'] = '#a0a0a0'
plt.rcParams['ytick.color'] = '#a0a0a0'
plt.rcParams['text.color'] = '#e0e0e0'
plt.rcParams['grid.color'] = '#2a2a4a'
plt.rcParams['grid.alpha'] = 0.5

## 1. Game Environment

Player I sees nothing before choosing (single information set). Player II sees one of three information sets:

| Info Set | Coin | Disclosure |
|:--------:|:----:|:-----------|
| $H_1$ | Heads | PI chose 1 |
| $H_2$ | Heads | PI chose 2 |
| $T$   | Tails | PI's choice hidden |

In [ ]:
PI_ACTIONS = [1, 2]
PII_ACTIONS = [3, 4]
REFEREE_OUTCOMES = [(1, 0.4), (2, 0.2), (3, 0.4)]
COIN_PROB = 0.5


class ParityGame:
    """Random Disclosure Parity Game environment.

    Episode flow:
      1. PI picks i ∈ {1, 2}
      2. Coin flip → heads / tails
      3. PII picks j ∈ {3, 4} (sees i only if heads)
      4. Referee draws r ∈ {1, 2, 3} with P = {0.4, 0.2, 0.4}
      5. Payoff to PI: +1 if (i + j + r) even, −1 if odd
    """

    def reset(self):
        self.pi_choice = None
        self.coin = None
        self.pii_choice = None
        self.ref_draw = None
        self.done = False
        self.payoff_pi = 0.0
        return self

    def pi_step(self, i: int):
        """Player I picks i ∈ {1, 2}, then the coin is flipped."""
        self.pi_choice = i
        self.coin = 'H' if random.random() < COIN_PROB else 'T'

    def get_pii_info_set(self) -> str:
        """Return PII's information set: 'H1', 'H2', or 'T'."""
        if self.coin == 'H':
            return f'H{self.pi_choice}'
        return 'T'

    def pii_step(self, j: int):
        """PII picks j ∈ {3, 4}, referee draws, payoff is resolved."""
        self.pii_choice = j
        r = random.random()
        self.ref_draw = 1 if r < 0.4 else (2 if r < 0.6 else 3)
        total = self.pi_choice + self.pii_choice + self.ref_draw
        self.payoff_pi = 1.0 if total % 2 == 0 else -1.0
        self.done = True


# Sanity check
game = ParityGame()
game.reset()
game.pi_step(1)
print(f"PI chose 1, coin = {game.coin}, PII info set = {game.get_pii_info_set()}")
game.pii_step(3)
print(f"PII chose 3, referee drew {game.ref_draw}, "
      f"sum = {game.pi_choice + game.pii_choice + game.ref_draw}, "
      f"payoff to PI = {game.payoff_pi:+.0f}")

## 2. Q-Learning Agents with Softmax Exploration

### Action selection via Boltzmann policy

For a maximiser (Player I) with temperature $\tau$:
$$\pi(a) = \frac{\exp(Q(a)/\tau)}{\sum_{a'} \exp(Q(a')/\tau)}$$

For a minimiser (Player II), we store $\tilde{Q} = -\text{payoff to PI}$, so higher $\tilde{Q}$ is better for PII. The same softmax formula applies to $\tilde{Q}$.

When Q-values are close (equilibrium requires mixing), softmax naturally produces near-uniform probabilities. When one Q-value dominates, the policy becomes nearly deterministic.

### Why track empirical frequencies?

In two-player games with mixed Nash equilibria, Q-values can **oscillate** as each player reacts to the other's shifting strategy. However, the **time-averaged play frequencies** converge to equilibrium. We track both.

In [ ]:
def softmax_probs(q_dict: dict, actions: list, tau: float) -> dict:
    """Compute Boltzmann probabilities from Q-values."""
    q = np.array([q_dict[a] for a in actions])
    q = q / tau
    q -= q.max()  # numerical stability
    e = np.exp(q)
    p = e / e.sum()
    return dict(zip(actions, p.tolist()))


class PIAgent:
    """Q-learning agent for Player I (maximiser).

    Single decision point — picks from {1, 2}.
    Q-table has one state with two actions.
    Uses softmax (Boltzmann) exploration.
    """

    def __init__(self, lr: float = 0.02, tau: float = 0.3):
        self.lr = lr
        self.tau = tau
        self.q = {a: 0.0 for a in PI_ACTIONS}
        self.action_counts = {a: 0 for a in PI_ACTIONS}
        self.total_episodes = 0

    def select_action(self, training: bool = True) -> int:
        probs = softmax_probs(self.q, PI_ACTIONS, self.tau)
        a = random.choices(PI_ACTIONS, weights=[probs[a] for a in PI_ACTIONS])[0]
        if training:
            self.action_counts[a] += 1
            self.total_episodes += 1
        return a

    def update(self, action: int, reward: float):
        self.q[action] += self.lr * (reward - self.q[action])

    def get_policy(self) -> dict:
        """Current softmax policy."""
        return softmax_probs(self.q, PI_ACTIONS, self.tau)

    def get_empirical_freqs(self) -> dict:
        """Time-averaged action frequencies."""
        total = sum(self.action_counts.values())
        if total == 0:
            return {a: 0.5 for a in PI_ACTIONS}
        return {a: self.action_counts[a] / total for a in PI_ACTIONS}


class PIIAgent:
    """Q-learning agent for Player II (minimiser).

    Three information sets: H1, H2, T.
    Q-table stores −payoff_to_PI (higher = better for PII).
    Uses softmax (Boltzmann) exploration.
    """

    INFO_SETS = ['H1', 'H2', 'T']

    def __init__(self, lr: float = 0.02, tau: float = 0.3):
        self.lr = lr
        self.tau = tau
        self.q = {s: {a: 0.0 for a in PII_ACTIONS} for s in self.INFO_SETS}
        self.action_counts = {s: {a: 0 for a in PII_ACTIONS} for s in self.INFO_SETS}

    def select_action(self, info_set: str, training: bool = True) -> int:
        probs = softmax_probs(self.q[info_set], PII_ACTIONS, self.tau)
        a = random.choices(PII_ACTIONS, weights=[probs[a] for a in PII_ACTIONS])[0]
        if training:
            self.action_counts[info_set][a] += 1
        return a

    def update(self, info_set: str, action: int, reward_pi: float):
        """Update with PII's reward (negated PI payoff)."""
        pii_reward = -reward_pi
        self.q[info_set][action] += self.lr * (pii_reward - self.q[info_set][action])

    def get_policy(self, info_set: str) -> dict:
        return softmax_probs(self.q[info_set], PII_ACTIONS, self.tau)

    def get_empirical_freqs(self, info_set: str) -> dict:
        total = sum(self.action_counts[info_set].values())
        if total == 0:
            return {a: 0.5 for a in PII_ACTIONS}
        return {a: self.action_counts[info_set][a] / total for a in PII_ACTIONS}

## 3. Training Loop

Each episode is a single game (PI chooses → coin flip → PII chooses → referee draws). Both agents update their Q-tables immediately with the terminal reward.

In [ ]:
def train_agents(
    pi_agent: PIAgent,
    pii_agent: PIIAgent,
    num_episodes: int = 500_000,
    log_interval: int = 2000
) -> dict:
    """Train both agents through self-play."""
    game = ParityGame()
    stats = {
        'episodes': [],
        'avg_payoff_pi': [],
        'pi_q1': [], 'pi_q2': [],
        'pii_qH1_3': [], 'pii_qH1_4': [],
        'pii_qH2_3': [], 'pii_qH2_4': [],
        'pii_qT_3': [], 'pii_qT_4': [],
        'pi_freq_1': [],
        'pii_T_freq_3': [],
    }
    payoff_buffer = []
    # Windowed action counts for tracking convergence
    window_pi = {a: 0 for a in PI_ACTIONS}
    window_pii_T = {a: 0 for a in PII_ACTIONS}

    for ep in tqdm(range(num_episodes), desc='Training'):
        game.reset()
        pi_action = pi_agent.select_action(training=True)
        game.pi_step(pi_action)
        info_set = game.get_pii_info_set()
        pii_action = pii_agent.select_action(info_set, training=True)
        game.pii_step(pii_action)

        payoff_pi = game.payoff_pi
        payoff_buffer.append(payoff_pi)
        pi_agent.update(pi_action, payoff_pi)
        pii_agent.update(info_set, pii_action, payoff_pi)

        window_pi[pi_action] += 1
        if info_set == 'T':
            window_pii_T[pii_action] += 1

        if (ep + 1) % log_interval == 0:
            stats['episodes'].append(ep + 1)
            stats['avg_payoff_pi'].append(np.mean(payoff_buffer[-log_interval:]))
            stats['pi_q1'].append(pi_agent.q[1])
            stats['pi_q2'].append(pi_agent.q[2])
            stats['pii_qH1_3'].append(pii_agent.q['H1'][3])
            stats['pii_qH1_4'].append(pii_agent.q['H1'][4])
            stats['pii_qH2_3'].append(pii_agent.q['H2'][3])
            stats['pii_qH2_4'].append(pii_agent.q['H2'][4])
            stats['pii_qT_3'].append(pii_agent.q['T'][3])
            stats['pii_qT_4'].append(pii_agent.q['T'][4])

            w_total = sum(window_pi.values())
            stats['pi_freq_1'].append(
                window_pi[1] / w_total if w_total > 0 else 0.5)
            t_total = sum(window_pii_T.values())
            stats['pii_T_freq_3'].append(
                window_pii_T[3] / t_total if t_total > 0 else 0.5)

            window_pi = {a: 0 for a in PI_ACTIONS}
            window_pii_T = {a: 0 for a in PII_ACTIONS}

    return stats

## 4. Training the Agents

In [ ]:
random.seed(42)
np.random.seed(42)

pi_agent = PIAgent(lr=0.02, tau=0.3)
pii_agent = PIIAgent(lr=0.02, tau=0.3)

NUM_EPISODES = 500_000

print("Random Disclosure Parity Game — Q-Learning with Softmax Exploration")
print("═" * 65)
print(f"  Episodes:       {NUM_EPISODES:,}")
print(f"  Learning rate:  {pi_agent.lr}")
print(f"  Temperature τ:  {pi_agent.tau}")
print(f"  PI actions:     {PI_ACTIONS}")
print(f"  PII actions:    {PII_ACTIONS}")
print(f"  P(heads):       {COIN_PROB}")
print(f"  Referee:        {{1,2,3}} with P={{0.4, 0.2, 0.4}}")
print(f"  Known value:    v = −0.30")

In [ ]:
stats = train_agents(pi_agent, pii_agent,
                     num_episodes=NUM_EPISODES, log_interval=2000)

## 5. Visualising Training Progress

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Random Disclosure Parity Game — Q-Learning Training',
             fontsize=16, fontweight='bold', color='#00d4aa')
ep = stats['episodes']

# (0,0) Average payoff
ax = axes[0, 0]
window = 15
rolling = np.convolve(stats['avg_payoff_pi'],
                      np.ones(window)/window, mode='valid')
ax.plot(ep[window-1:], rolling, color='#ffd93d', linewidth=1.2,
        label='Rolling avg')
ax.axhline(y=-0.3, color='#ff6b6b', linestyle='--', linewidth=1.5,
           label='Theory: v = −0.30')
ax.set_title('Average Payoff to PI', color='#00d4aa')
ax.set_xlabel('Episode')
ax.set_ylabel('Payoff')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# (0,1) PI Q-values
ax = axes[0, 1]
ax.plot(ep, stats['pi_q1'], color='#ff6b6b', alpha=0.7,
        linewidth=0.8, label='Q(1)')
ax.plot(ep, stats['pi_q2'], color='#4ecdc4', alpha=0.7,
        linewidth=0.8, label='Q(2)')
ax.axhline(y=-0.3, color='#ffd93d', linestyle='--', linewidth=1,
           alpha=0.5, label='v* = −0.30')
ax.set_title('Player I Q-Values', color='#00d4aa')
ax.set_xlabel('Episode')
ax.set_ylabel('Q')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# (0,2) PI empirical freq
ax = axes[0, 2]
ax.plot(ep, stats['pi_freq_1'], color='#a29bfe', linewidth=0.8,
        alpha=0.7, label='P(action=1) windowed')
ax.axhline(y=0.5, color='#ffd93d', linestyle='--', linewidth=1.5,
           label='Equilibrium: 0.50')
ax.set_title('PI Action Frequency (windowed)', color='#00d4aa')
ax.set_xlabel('Episode')
ax.set_ylabel('Freq(action=1)')
ax.set_ylim(0, 1)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# (1,0) PII Q-values at H1
ax = axes[1, 0]
ax.plot(ep, stats['pii_qH1_3'], color='#ff6b6b', alpha=0.7,
        linewidth=0.8, label='Q(H1, 3)')
ax.plot(ep, stats['pii_qH1_4'], color='#4ecdc4', alpha=0.7,
        linewidth=0.8, label='Q(H1, 4)')
ax.set_title('PII Q-Values at H₁', color='#00d4aa')
ax.set_xlabel('Episode')
ax.set_ylabel('Q (higher = better for PII)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# (1,1) PII Q-values at H2
ax = axes[1, 1]
ax.plot(ep, stats['pii_qH2_3'], color='#ff6b6b', alpha=0.7,
        linewidth=0.8, label='Q(H2, 3)')
ax.plot(ep, stats['pii_qH2_4'], color='#4ecdc4', alpha=0.7,
        linewidth=0.8, label='Q(H2, 4)')
ax.set_title('PII Q-Values at H₂', color='#00d4aa')
ax.set_xlabel('Episode')
ax.set_ylabel('Q (higher = better for PII)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# (1,2) PII Q-values at T + empirical freq
ax = axes[1, 2]
ax.plot(ep, stats['pii_qT_3'], color='#ff6b6b', alpha=0.5,
        linewidth=0.7, label='Q(T, 3)')
ax.plot(ep, stats['pii_qT_4'], color='#4ecdc4', alpha=0.5,
        linewidth=0.7, label='Q(T, 4)')
ax2 = ax.twinx()
ax2.plot(ep, stats['pii_T_freq_3'], color='#a29bfe', alpha=0.6,
         linewidth=0.8, label='Freq(3) windowed')
ax2.axhline(y=0.5, color='#ffd93d', linestyle='--', linewidth=1)
ax2.set_ylabel('Freq(action=3)', color='#a29bfe')
ax2.set_ylim(0, 1)
ax.set_title('PII at T: Q-Values & Frequency', color='#00d4aa')
ax.set_xlabel('Episode')
ax.set_ylabel('Q (higher = better for PII)')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=7, loc='upper right')

plt.tight_layout()
plt.show()

## 6. Analysing Learned Strategies

We examine both the **instantaneous Q-values** (which may oscillate) and the **cumulative empirical frequencies** (which converge to the Nash equilibrium).

In [ ]:
print("═" * 65)
print("  LEARNED Q-VALUES AND STRATEGIES")
print("═" * 65)

print("\n── Player I (maximiser) ──")
for a in PI_ACTIONS:
    print(f"    Q(action={a}) = {pi_agent.q[a]:+.4f}")
pi_policy = pi_agent.get_policy()
pi_freq = pi_agent.get_empirical_freqs()
print(f"  Softmax policy:      P(1) = {pi_policy[1]:.3f},  P(2) = {pi_policy[2]:.3f}")
print(f"  Empirical frequency: P(1) = {pi_freq[1]:.3f},  P(2) = {pi_freq[2]:.3f}")
print(f"  Equilibrium:         P(1) = 0.500,  P(2) = 0.500")

print("\n── Player II (minimiser) ──")
print("  Q stores −payoff_PI, so higher Q = better for PII.")
for info_set in ['H1', 'H2', 'T']:
    print(f"\n  Info set {info_set}:")
    for a in PII_ACTIONS:
        print(f"    Q({info_set}, {a}) = {pii_agent.q[info_set][a]:+.4f}")
    pol = pii_agent.get_policy(info_set)
    freq = pii_agent.get_empirical_freqs(info_set)
    print(f"    Softmax policy:  P(3) = {pol[3]:.3f},  P(4) = {pol[4]:.3f}")
    print(f"    Empirical freq:  P(3) = {freq[3]:.3f},  P(4) = {freq[4]:.3f}")

print("\n── Summary ──")
print("  H1: should prefer 3 (match PI=1 parity) → "
      f"learned prefers {max(PII_ACTIONS, key=lambda a: pii_agent.q['H1'][a])}")
print("  H2: should prefer 4 (match PI=2 parity) → "
      f"learned prefers {max(PII_ACTIONS, key=lambda a: pii_agent.q['H2'][a])}")
t_freq = pii_agent.get_empirical_freqs('T')
print(f"  T:  should be ~50/50 → empirical freq(3) = {t_freq[3]:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle('Learned Q-Values by Decision Point',
             fontsize=15, fontweight='bold', color='#00d4aa')
colors = ['#ff6b6b', '#4ecdc4']

# Player I
ax = axes[0]
vals = [pi_agent.q[a] for a in PI_ACTIONS]
bars = ax.bar([str(a) for a in PI_ACTIONS], vals,
              color=colors, edgecolor='#4a4a6a')
ax.set_title('Player I', color='#00d4aa', fontweight='bold')
ax.set_xlabel('Action')
ax.set_ylabel('Q-value')
ax.axhline(y=-0.3, color='#ffd93d', linestyle='--', linewidth=1, label='v*=−0.30')
ax.legend(fontsize=8)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2,
            v + 0.02 * (-1 if v < 0 else 1),
            f'{v:+.3f}', ha='center',
            va='top' if v < 0 else 'bottom', fontsize=9)

# PII at each info set
for idx, info_set in enumerate(['H1', 'H2', 'T']):
    ax = axes[idx + 1]
    vals = [pii_agent.q[info_set][a] for a in PII_ACTIONS]
    bars = ax.bar([str(a) for a in PII_ACTIONS], vals,
                  color=colors, edgecolor='#4a4a6a')
    ax.set_title(f'Player II @ {info_set}',
                 color='#00d4aa', fontweight='bold')
    ax.set_xlabel('Action')
    ax.set_ylabel('Q (higher = better for PII)')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                v + 0.02 * (-1 if v < 0 else 1),
                f'{v:+.3f}', ha='center',
                va='top' if v < 0 else 'bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 7. Theoretical Optimal Strategy

From the minimax analysis (see `src/minimax/random_disclosure_parity_game_demo.py`), the conditional payoffs to Player I are:

| $g(i,j)$ | $j=3$ | $j=4$ |
|:---------:|:-----:|:-----:|
| $i=1$ | $-0.6$ | $+0.6$ |
| $i=2$ | $+0.6$ | $-0.6$ |

Player II's optimal strategy:
- **$H_1$ (knows PI=1)**: play 3 → payoff $g(1,3) = -0.6$ (bad for PI)
- **$H_2$ (knows PI=2)**: play 4 → payoff $g(2,4) = -0.6$ (bad for PI)
- **$T$ (uninformed)**: randomise 50/50 → expected payoff = 0

Game value: $v = \frac{1}{2}(-0.6) + \frac{1}{2}(0) = -0.30$

In [ ]:
def g(i: int, j: int) -> float:
    """Conditional payoff to PI for choices (i, j), averaged over referee."""
    return sum(p * (1 if (i + j + r) % 2 == 0 else -1)
               for r, p in REFEREE_OUTCOMES)


print("═" * 65)
print("  COMPARISON: LEARNED vs THEORETICAL OPTIMAL")
print("═" * 65)

print("\n── Conditional payoffs g(i, j) ──")
print(f"  g(1,3) = {g(1,3):+.1f}    g(1,4) = {g(1,4):+.1f}")
print(f"  g(2,3) = {g(2,3):+.1f}    g(2,4) = {g(2,4):+.1f}")

print("\n── Player II policy ──")
optimal_pii = {'H1': 3, 'H2': 4, 'T': 'mix 50/50'}
for info_set in ['H1', 'H2', 'T']:
    learned = max(PII_ACTIONS, key=lambda a: pii_agent.q[info_set][a])
    opt = optimal_pii[info_set]
    freq = pii_agent.get_empirical_freqs(info_set)
    if info_set == 'T':
        ok = abs(freq[3] - 0.5) < 0.1
        print(f"  {info_set}: empirical P(3)={freq[3]:.3f}  "
              f"(optimal: {opt})  {'✓' if ok else '~'}")
    else:
        ok = learned == opt
        print(f"  {info_set}: learned={learned}  "
              f"(optimal: {opt})  {'✓' if ok else '✗'}")

print("\n── Player I policy ──")
pi_freq = pi_agent.get_empirical_freqs()
ok_pi = abs(pi_freq[1] - 0.5) < 0.1
print(f"  Empirical P(1)={pi_freq[1]:.3f}  "
      f"(optimal: 0.500)  {'✓' if ok_pi else '~'}")

## 8. Evaluation: Learned Agents vs Optimal Play

We test the trained agents against opponents playing the known optimal strategy.

In [ ]:
class OptimalPIAgent:
    """Optimal Player I: mixes {1, 2} uniformly."""
    def select_action(self, training=False) -> int:
        return random.choice(PI_ACTIONS)


class OptimalPIIAgent:
    """Optimal Player II: match parity when informed, randomise when not."""
    def select_action(self, info_set: str, training=False) -> int:
        if info_set == 'H1':
            return 3
        elif info_set == 'H2':
            return 4
        return random.choice(PII_ACTIONS)


class EmpiricalAgent:
    """Replay the learned empirical strategy (stochastic)."""

    def __init__(self, pi_agent: PIAgent = None, pii_agent: PIIAgent = None):
        self._pi = pi_agent
        self._pii = pii_agent

    def select_action_pi(self, training=False) -> int:
        freq = self._pi.get_empirical_freqs()
        return random.choices(PI_ACTIONS,
                              weights=[freq[a] for a in PI_ACTIONS])[0]

    def select_action_pii(self, info_set: str, training=False) -> int:
        freq = self._pii.get_empirical_freqs(info_set)
        return random.choices(PII_ACTIONS,
                              weights=[freq[a] for a in PII_ACTIONS])[0]


def evaluate(pi_select, pii_select, num_games=100_000) -> float:
    """Evaluate by playing num_games. Returns avg payoff to PI."""
    game = ParityGame()
    total = 0.0
    for _ in range(num_games):
        game.reset()
        i = pi_select()
        game.pi_step(i)
        info = game.get_pii_info_set()
        j = pii_select(info)
        game.pii_step(j)
        total += game.payoff_pi
    return total / num_games


random.seed(123)
opt_pi = OptimalPIAgent()
opt_pii = OptimalPIIAgent()
emp = EmpiricalAgent(pi_agent, pii_agent)

print("═" * 65)
print("  EVALUATION (100,000 games per matchup)")
print("═" * 65)

matchups = [
    ('Learned PI vs Learned PII',
     lambda: pi_agent.select_action(False),
     lambda s: pii_agent.select_action(s, False)),
    ('Empirical PI vs Empirical PII',
     emp.select_action_pi,
     emp.select_action_pii),
    ('Learned PI vs Optimal PII',
     lambda: pi_agent.select_action(False),
     lambda s: opt_pii.select_action(s)),
    ('Optimal PI vs Learned PII',
     lambda: opt_pi.select_action(),
     lambda s: pii_agent.select_action(s, False)),
    ('Optimal PI vs Optimal PII',
     lambda: opt_pi.select_action(),
     lambda s: opt_pii.select_action(s)),
]

results = []
for label, pi_fn, pii_fn in matchups:
    avg = evaluate(pi_fn, pii_fn)
    results.append((label, avg))
    print(f"  {label:>40s}: {avg:+.4f}")

print(f"\n  Theoretical game value: −0.3000")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

labels = [r[0] for r in results]
values = [r[1] for r in results]
bar_colors = ['#ff6b6b' if v < -0.3 else '#4ecdc4' if v > -0.3 else '#ffd93d'
              for v in values]

bars = ax.barh(labels, values, color=bar_colors,
               edgecolor='#4a4a6a', height=0.5)
ax.axvline(x=-0.3, color='#ffd93d', linestyle='--', linewidth=2,
           label='Game value (−0.30)')
ax.axvline(x=0, color='#4a4a6a', linestyle='-', linewidth=0.5)
ax.set_xlabel('Average Payoff to Player I ($)')
ax.set_title('Evaluation: Learned vs Optimal Agents',
             fontsize=14, fontweight='bold', color='#00d4aa')
ax.legend(loc='lower right')
ax.set_xlim(-0.7, 0.15)

for bar, v in zip(bars, values):
    offset = -0.02 if v < 0 else 0.02
    ax.text(v + offset, bar.get_y() + bar.get_height()/2,
            f'{v:+.3f}', ha='right' if v < 0 else 'left',
            va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## 9. Watching Sample Games

In [ ]:
def play_verbose(pi_fn, pii_fn, pi_name='PI', pii_name='PII', n=10):
    """Play n games and print each step."""
    game = ParityGame()
    for k in range(n):
        game.reset()
        i = pi_fn()
        game.pi_step(i)
        info = game.get_pii_info_set()
        j = pii_fn(info)
        game.pii_step(j)
        s = game.pi_choice + game.pii_choice + game.ref_draw
        par = 'even' if s % 2 == 0 else 'odd'
        winner = pi_name if game.payoff_pi > 0 else pii_name
        print(f"  Game {k+1:2d}: {pi_name}={i}, coin={game.coin}, "
              f"info={info}, {pii_name}={j}, r={game.ref_draw}, "
              f"sum={s} ({par}) → {winner} wins")


random.seed(99)
print("═" * 65)
print("  SAMPLE GAMES: Learned PI vs Learned PII")
print("═" * 65)
play_verbose(lambda: pi_agent.select_action(False),
             lambda s: pii_agent.select_action(s, False),
             'Learned PI', 'Learned PII')

random.seed(99)
print(f"\n{'═' * 65}")
print("  SAMPLE GAMES: Optimal PI vs Optimal PII")
print("═" * 65)
play_verbose(lambda: opt_pi.select_action(),
             lambda s: opt_pii.select_action(s),
             'Optimal PI', 'Optimal PII')

## 10. Sensitivity Analysis: Learning Rate and Temperature

We explore how learning rate $\alpha$ and softmax temperature $\tau$ affect convergence.

In [ ]:
configs = [
    {'lr': 0.005, 'tau': 0.3, 'label': 'α=0.005, τ=0.3'},
    {'lr': 0.02,  'tau': 0.3, 'label': 'α=0.02,  τ=0.3'},
    {'lr': 0.10,  'tau': 0.3, 'label': 'α=0.10,  τ=0.3'},
    {'lr': 0.02,  'tau': 0.1, 'label': 'α=0.02,  τ=0.1'},
    {'lr': 0.02,  'tau': 1.0, 'label': 'α=0.02,  τ=1.0'},
]

sensitivity_results = []

for cfg in configs:
    random.seed(42)
    np.random.seed(42)
    pa = PIAgent(lr=cfg['lr'], tau=cfg['tau'])
    pb = PIIAgent(lr=cfg['lr'], tau=cfg['tau'])
    st = train_agents(pa, pb, num_episodes=300_000, log_interval=2000)
    sensitivity_results.append((cfg['label'], st, pa, pb))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Sensitivity Analysis',
             fontsize=15, fontweight='bold', color='#00d4aa')
cmap = ['#ff6b6b', '#4ecdc4', '#a29bfe', '#ffd93d', '#74b9ff']

# Average payoff convergence
ax = axes[0]
for (label, st, _, _), col in zip(sensitivity_results, cmap):
    w = 15
    roll = np.convolve(st['avg_payoff_pi'], np.ones(w)/w, mode='valid')
    ax.plot(st['episodes'][w-1:], roll, color=col,
            linewidth=1.2, alpha=0.8, label=label)
ax.axhline(y=-0.3, color='white', linestyle='--', linewidth=1.5,
           alpha=0.5, label='v* = −0.30')
ax.set_xlabel('Episode')
ax.set_ylabel('Avg Payoff to PI (smoothed)')
ax.set_title('Payoff Convergence', color='#00d4aa')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# PII T-freq convergence
ax = axes[1]
for (label, st, _, _), col in zip(sensitivity_results, cmap):
    w = 15
    roll = np.convolve(st['pii_T_freq_3'], np.ones(w)/w, mode='valid')
    ax.plot(st['episodes'][w-1:], roll, color=col,
            linewidth=1.2, alpha=0.8, label=label)
ax.axhline(y=0.5, color='white', linestyle='--', linewidth=1.5,
           alpha=0.5, label='Equilibrium: 0.50')
ax.set_xlabel('Episode')
ax.set_ylabel('PII Freq(3) at T (smoothed)')
ax.set_title('PII Mixing at T', color='#00d4aa')
ax.set_ylim(0, 1)
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Conclusion

### Summary

We trained two Q-learning agents (with Boltzmann exploration) on the **Random Disclosure Parity Game** — a zero-sum game with imperfect information and chance moves.

**Key findings:**

1. **Player II's informed strategy**: The agent correctly learned to **match parity** — playing 3 at $H_1$ (when informed PI chose 1) and 4 at $H_2$ (when informed PI chose 2). The Q-value separation is large and stable.

2. **Player II's uninformed strategy**: At information set $T$, the Q-values oscillate (neither action is consistently better), and the empirical action frequency converges to approximately **50/50** — the optimal uniform randomisation.

3. **Player I's indifference**: Player I's empirical action frequency converges to approximately **50/50**, reflecting the theoretical result that PI is indifferent at equilibrium. Note that the instantaneous Q-values may differ, but the time-averaged behaviour is correct.

4. **Game value**: The average payoff converges to approximately **$-0.30$**, matching the analytical game value $v = -0.6 \times P(\text{heads}) = -0.30$.

### Why Softmax Matters

Standard $\varepsilon$-greedy Q-learning converges to deterministic policies, which cannot represent the mixed-strategy equilibrium required at $T$ and for Player I. Boltzmann (softmax) exploration with fixed temperature $\tau$ naturally produces stochastic action selection:

- When Q-values **diverge** (at $H_1$, $H_2$), softmax converges to nearly deterministic play ✓
- When Q-values **oscillate around parity** (at $T$, for PI), softmax produces approximately uniform mixing ✓

The key insight: in games with mixed Nash equilibria, **time-averaged empirical frequencies** are the correct measure of learned strategy, not instantaneous Q-value comparisons.

### Comparison with the Addition Game

| Feature | Addition Game | Parity Game |
|:--------|:-------------|:------------|
| Information | Perfect | Imperfect (coin toss) |
| Game type | Sequential, multi-step | Single-round with disclosure |
| State space | Sum $\in \{0, \ldots, N\}$ | Info sets $\{H_1, H_2, T\}$ |
| Optimal strategy | Deterministic | Mixed (randomise at $T$) |
| Game value | $\pm 1$ | $-0.30$ |
| Q-learning variant | $\varepsilon$-greedy | Softmax (Boltzmann) |
| Challenge | Multi-step credit assignment | Learning to randomise |